# Test 3: inferencia

[Guía de la entrega](../../../../docs/README.md) · [Lectura conjunta](study-overview.ipynb) · [Caso e historial](../README.md)

**Objetivo:** encontrar una configuración de generación que resuelva el caso con instrucciones manuales fijas.

**Negociador:** llamada al modelo que responde al cliente. **Política H:** instrucciones que guían esa respuesta. Aquí H está escrita manualmente y no cambia.

Las celdas leen evidencia guardada; no llaman al modelo.

## Alcance del control

- **Entrada:** tabla de deudas y conversación previa, no historial de pagos. La capacidad de pago es el dinero disponible cada mes para la deuda; no es el ingreso.
- **Fijo:** política manual, pesos del modelo y evaluador Python.
- **Ajuste:** temperatura; después, perfil con razonamiento.
- **Salida:** decisión con acción, deuda, cantidad de pagos y monto por cuota.

No se generan políticas nuevas ni instrucciones de mejora en este control.

![Control de inferencia](../diagrams/inference-control.svg)

In [1]:
from pathlib import Path
import hashlib
import html
import json
from IPython.display import HTML, display

relative = Path("research/cases/case-0003-debt-negotiation")
case_dir = next(base / relative for base in [Path.cwd(), *Path.cwd().parents]
                if (base / relative / "results").is_dir())
run_names = ['20261106T202558Z-inference-qwen3-8b-tuning', '20261106T203000Z-inference-qwen3-32b-tuning', '20261106T204139Z-inference-qwen3-8b-thinking-tuning', '20261106T204259Z-inference-qwen3-8b-thinking-confirmation']

def load_record(name):
    folder = case_dir / "results" / name
    manifest = json.loads((folder / "manifest.json").read_text())
    for filename, digest in manifest.items():
        assert Path(filename).name == filename
        assert hashlib.sha256((folder / filename).read_bytes()).hexdigest() == digest
    return {"name": name, "protocol": json.loads((folder / "protocol.json").read_text()),
            "report": json.loads((folder / "report.json").read_text())}

records = [load_record(name) for name in run_names]
def table(headers, rows):
    head = "".join(f'<th style="text-align:left;padding:8px;border-bottom:2px solid #667085">{html.escape(str(h))}</th>' for h in headers)
    body = "".join("<tr>" + "".join(f'<td style="text-align:left;vertical-align:top;overflow-wrap:anywhere;padding:8px;border-bottom:1px solid #ddd">{html.escape(str(v))}</td>' for v in row) + "</tr>" for row in rows)
    display(HTML(f'<table style="width:100%;border-collapse:collapse;margin:16px 0"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))
print(f"Integridad verificada: {len(records)} ejecuciones locales.")

def model_label(item):
    return item["protocol"]["model_key"] + (" · razonamiento" if item["protocol"].get("enable_thinking") else " · directo")

Integridad verificada: 4 ejecuciones locales.


## Política y respuesta

**System prompt:** contrato fijo + política manual + esquema JSON.
**Entrada (user):** tabla de deudas e historial. Los valores esperados solo los ve el evaluador.

<details>
<summary><strong>Política manual completa</strong></summary>

```text
Aplica estas reglas en orden usando el historial completo y la tabla de deudas:
1. Si el cliente pide terminar o rechaza toda negociación, responde stop.
   Rechazar una oferta concreta no significa rechazar toda negociación.
2. Identifica la capacidad mensual más reciente. Si no está indicada, responde ask.
   No la supongas igual a cero ni vuelvas a preguntarla si ya se conoce.
3. Si el cliente acepta explícitamente una deuda y un plan ofrecidos, confirma con accept
   solo ese plan, siempre que su cuota respete la capacidad. Una propuesta del asistente
   por sí sola no constituye aceptación.
4. Si no hay aceptación, recorre las deudas de mayor a menor monto. Para cada deuda,
   prueba primero un pago: cuota = monto. Si supera la capacidad mensual, prueba dos
   pagos: cuota = monto / 2. La cuota debe ser menor o igual a la capacidad.
   En cuanto encuentres un plan viable, responde offer con esa deuda, cantidad de pagos
   y cuota calculada. No elijas una deuda menor si una mayor cabe en dos pagos.
5. Si ninguna deuda admite uno o dos pagos dentro de la capacidad, responde stop.
Para ask y stop: debt=null, payments=0, installment=0.
Para offer y accept: debt identifica la deuda; payments indica 1 o 2;
installment es el monto de la deuda dividido entre payments, no la capacidad mensual.
No inventes descuentos, no ejecutes pagos ni alteres la tabla de deudas.
```

</details>

| Campo | Significado | Valores |
|---|---|---|
| `action` | Acción elegida | `ask`: preguntar; `offer`: ofrecer; `accept`: confirmar aceptación; `stop`: detenerse |
| `debt` | Identificador de deuda | A=200; B=300; C=600; `null` si pregunta o se detiene |
| `payments` | Cantidad de pagos | 1 o 2; cero para `ask`/`stop` |
| `installment` | Monto de cada cuota | Deuda dividida entre pagos; cero para `ask`/`stop` |

**Respuesta esperada** con capacidad de pago mensual de 225:

```json
{
  "action": "offer",
  "debt": "B",
  "payments": 2,
  "installment": 150
}
```

B es la mayor deuda viable: dos cuotas de 150 no superan 225. La deuda C requiere al menos 300 por cuota.
Python redacta el mensaje; no corrige los valores elegidos por el modelo.

## 1. Parámetros de inferencia

Se ajusta **cómo el modelo genera respuestas**, sin entrenarlo. Un token es una unidad de texto del modelo; no equivale siempre a una palabra.

| Parámetro | Significado | Valores y fundamento |
|:---|:---|:---|
| Temperatura | Regula la diversidad al elegir el siguiente token | **0:** elegir el más probable.<br>**0,3:** explorar poca variación.<br>**0,7:** referencia sin razonamiento.<br>**0,6:** referencia con razonamiento. |
| `top_k` | Limita la selección a los k tokens más probables | **20:** referencia de Qwen; fijo durante el barrido. |
| `top_p` | Filtra candidatos por probabilidad acumulada | **0,8:** sin razonamiento.<br>**0,95:** con razonamiento. |
| `max_tokens` | Tope de salida por llamada, incluido razonamiento | **512:** barrido directo.<br>**2.048:** con razonamiento.<br>**4.096:** comprobar cortes de salida. |

**Referencia:** [Qwen3](https://huggingface.co/Qwen/Qwen3-8B#best-practices). Temperaturas 0 y 0,3: exploratorias. Topes de tokens: límites locales de prueba.

**Barrido directo:** 12 casos × (una repetición a temperatura 0 + tres a 0,3 + tres a 0,7) = **84 llamadas por modelo**.
El perfil posterior con razonamiento se prueba una vez en los 12 casos antes de confirmar.

## 2. Criterio de aceptación

Sea **N** el número de respuestas evaluadas:

$$Q=\frac{\text{respuestas correctas sin infracciones}}{N},\qquad
V=\frac{\text{respuestas con al menos una infracción}}{N}.$$

- **Respuesta correcta:** coincide con la referencia en:
  - Acción: preguntar, ofrecer, aceptar o detenerse.
  - Deuda elegida.
  - Cantidad de pagos.
  - Monto de cada cuota.

- **Infracciones:** se registra alguna de estas condiciones:
  - Salida inválida o incompleta.
  - Plan no permitido.
  - Cuota mal calculada o superior a la capacidad de pago.
  - Oferta tras un rechazo explícito a negociar.
  - Aceptación sin consentimiento.

- **Q ≥ 80 %:** respuestas correctas y sin infracciones:
  - Con 8 respuestas: al menos **7 aciertos**.
  - Con 24 respuestas: al menos **20 aciertos**.

- **V = 0:** ninguna respuesta evaluada infringe las reglas.
  - Una cuota impagable o una aceptación inventada invalida la decisión.

**Ejemplo:** ofrecer A cuando B también cabe:
- La elección es incorrecta: reduce Q.
- El plan puede respetar las restricciones: no necesariamente aumenta V.

**Alcance:** 80 % es una meta provisional del piloto. V = 0 describe las respuestas evaluadas, no todos los casos posibles.

**Selección:** priorizar perfiles sin infracciones; después, mayor Q, menor V y menor temperatura. La confirmación usa el perfil seleccionado sin reajustarlo.

In [2]:
rows = []
for item in records:
    if item["protocol"]["phase"] != "tuning":
        continue
    for profile in item["report"].get("profiles", []):
        rows.append([model_label(item), profile["sampling"]["temperature"],
                     f'{profile["successes"]}/{profile["count"]}', f'{profile["q"]:.1%}',
                     f'{profile["v"]:.1%}', "Cumple" if profile["passes"] else "No cumple"])
table(["Modelo", "Temperatura", "Aciertos", "Q", "V", "Criterio piloto"], rows)

Modelo,Temperatura,Aciertos,Q,V,Criterio piloto
qwen3-8b · directo,0.0,6/12,50.0%,0.0%,No cumple
qwen3-8b · directo,0.3,18/36,50.0%,0.0%,No cumple
qwen3-8b · directo,0.7,18/36,50.0%,0.0%,No cumple
qwen3-32b · directo,0.0,8/12,66.7%,33.3%,No cumple
qwen3-32b · directo,0.3,24/36,66.7%,27.8%,No cumple
qwen3-32b · directo,0.7,24/36,66.7%,27.8%,No cumple
qwen3-8b · razonamiento,0.6,12/12,100.0%,0.0%,Cumple


## 3. Repeticiones y consumo

Un **escenario** contiene una conversación simulada y su decisión esperada.

| Grupo del Test 3 | Uso |
|:---|:---|
| **12 conocidos** | Conversaciones ya usadas en diagnóstico.<br>Incluyen capacidad de pago, planes, rechazo y consentimiento.<br>Permiten comparar perfiles; no entrenan pesos. |
| **8 reservados** | Conversaciones distintas, fijadas antes del barrido.<br>No intervienen en la selección del perfil.<br>Permiten comprobarlo después. |
| **3 repeticiones** | Misma conversación y perfil; distintas semillas.<br>**8 casos × 3 repeticiones = 24 respuestas.**<br>Son ocho casos, no 24 casos diferentes. |

- **Semilla:** inicializa el muestreo aleatorio para repetir la configuración.
- **Valores:** 11, 22 y 33; identifican repeticiones, no se optimizaron.
- **Uso posterior:** los ocho casos se reutilizaron en otros diagnósticos; ya no eran una prueba nueva.

El tiempo incluye carga, no descarga. El pico MLX mide memoria asignada por el framework, no toda la RAM.

In [3]:
rows = []
for item in records:
    for result in item["report"]["evaluations"]:
        rows.append([model_label(item), item["protocol"]["phase"], result["sampling"]["temperature"],
                     result["sampling"]["seed"], f'{result["q"]:.1%}', f'{result["v"]:.1%}'])
table(["Modelo", "Etapa", "Temperatura", "Semilla", "Q", "V"], rows)
table(["Modelo", "Etapa", "Llamadas", "Tokens", "Segundos", "Pico MLX GB"],
      [[model_label(r), r["protocol"]["phase"], r["report"]["calls"], r["report"]["tokens"],
        round(r["report"]["elapsed_seconds"],1), round(r["report"]["peak_mlx_gb"] or 0,2)] for r in records])

Modelo,Etapa,Temperatura,Semilla,Q,V
qwen3-8b · directo,tuning,0.0,11,50.0%,0.0%
qwen3-8b · directo,tuning,0.3,11,50.0%,0.0%
qwen3-8b · directo,tuning,0.3,22,50.0%,0.0%
qwen3-8b · directo,tuning,0.3,33,50.0%,0.0%
qwen3-8b · directo,tuning,0.7,11,50.0%,0.0%
qwen3-8b · directo,tuning,0.7,22,50.0%,0.0%
qwen3-8b · directo,tuning,0.7,33,50.0%,0.0%
qwen3-32b · directo,tuning,0.0,11,66.7%,33.3%
qwen3-32b · directo,tuning,0.3,11,66.7%,25.0%
qwen3-32b · directo,tuning,0.3,22,66.7%,25.0%


Modelo,Etapa,Llamadas,Tokens,Segundos,Pico MLX GB
qwen3-8b · directo,tuning,84,70154,105.4,5.43
qwen3-32b · directo,tuning,84,70673,539.1,19.35
qwen3-8b · razonamiento,tuning,12,15110,65.4,5.42
qwen3-8b · razonamiento,confirmation,24,31036,178.3,5.42


## 4. Casos de confirmación

- **Cinco capacidades de pago:** 110, 175, 310, 650 y 90.
- **Tres situaciones:** capacidad desconocida, rechazo explícito y aceptación de un plan.
- **Repetición:** cada conversación se evalúa tres veces con la misma política; se observan posibles cambios de respuesta por el muestreo.

No se ajusta el perfil con estos resultados dentro del Test 3.

In [4]:
confirmed = [r for r in records if r["protocol"]["phase"] == "confirmation"]
rows = []
for item in confirmed:
    result = item["report"]["confirmation"]
    rows.append([model_label(item), f'{sum(r["success"] for r in result["rows"])}/{len(result["rows"])}',
                 f'{result["q"]:.1%}', f'{result["v"]:.1%}', "Cumple" if item["report"]["passes"] else "No cumple"])
table(["Modelo", "Aciertos", "Q", "V", "Criterio piloto"], rows)
if not confirmed:
    print("No se llevó ningún perfil local a confirmación: ninguno cumplió el criterio de ajuste.")
for item in confirmed:
    rows = []
    for case in item["protocol"]["cases"]:
        outputs = [row for result in item["report"]["evaluations"] for row in result["rows"] if row["id"] == case["id"]]
        expected = " / ".join(str(case["expected"][k]) for k in ("action","debt","payments","installment"))
        rows.append([case["id"], expected, f'{sum(r["success"] for r in outputs)}/{len(outputs)}',
                     "; ".join(sorted({json.dumps(r["raw_reply"],ensure_ascii=False) for r in outputs}))])
    table(["Caso", "Esperado: acción/deuda/pagos/cuota", "Aciertos", "Respuestas originales"], rows)

Modelo,Aciertos,Q,V,Criterio piloto
qwen3-8b · razonamiento,24/24,100.0%,0.0%,Cumple


Caso,Esperado: acción/deuda/pagos/cuota,Aciertos,Respuestas originales
confirmation-110,offer / A / 2 / 100,3/3,"{""action"": ""offer"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100}"
confirmation-175,offer / B / 2 / 150,3/3,"{""action"": ""offer"", ""debt"": ""B"", ""payments"": 2, ""installment"": 150}"
confirmation-310,offer / C / 2 / 300,3/3,"{""action"": ""offer"", ""debt"": ""C"", ""payments"": 2, ""installment"": 300}"
confirmation-650,offer / C / 1 / 600,3/3,"{""action"": ""offer"", ""debt"": ""C"", ""payments"": 1, ""installment"": 600}"
confirmation-90,stop / None / 0 / 0,3/3,"{""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0}"
confirmation-unknown,ask / None / 0 / 0,3/3,"{""action"": ""ask"", ""debt"": null, ""payments"": 0, ""installment"": 0}"
confirmation-decline,stop / None / 0 / 0,3/3,"{""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0}"
confirmation-consent,accept / A / 2 / 100,3/3,"{""action"": ""accept"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100}"


## 5. Control por API

`gpt-4o-mini-2024-07-18`: muestreo predeterminado y temperatura cero. Misma política, casos y evaluador, con JSON estructurado. El mecanismo de formato difiere de MLX; no se aísla el efecto del modelo.

Se comprueban 12 casos conocidos. Solo si cumplen, se evalúan ocho reservados tres veces. El prompt permanece fijo.


In [5]:
api_records = [load_record(name) for name in ['20261106T203920Z-api-reference', '20261106T204025Z-api-reference']]
for item in api_records:
    report = item['report']
    table(['Ejecución', 'Temperatura'], [[item['name'], item['protocol']['config'].get('temperature', 'Predeterminada')]])
    table(['Etapa', 'Repetición', 'Aciertos', 'Q', 'V'],
          [[{'known':'Conocidos', 'confirmation':'Confirmación'}[r['phase']], r['repeat']+1, f"{sum(x['success'] for x in r['rows'])}/{len(r['rows'])}",
            f"{r['q']:.1%}", f"{r['v']:.1%}"] for r in report['evaluations']])
    table(['Estado', 'Criterio completo', 'Llamadas', 'Tokens', 'Costo estimado USD'],
          [[{'completed':'Terminada', 'stopped':'Detenida'}.get(report['status'],report['status']), 'Cumple' if report.get('passes') else 'No cumple', report['calls'], report['tokens'],
            f"{report['estimated_cost_usd']:.6f}"]])
    rows = []
    for case in item['protocol']['confirmation_cases']:
        outputs = [row for r in report['evaluations'] if r['phase']=='confirmation' for row in r['rows'] if row['id']==case['id']]
        if not outputs: continue
        rows.append([case['id'], json.dumps(case['expected'],ensure_ascii=False),
                     f"{sum(r['success'] for r in outputs)}/{len(outputs)}",
                     '; '.join(sorted({json.dumps(r['raw_reply'],ensure_ascii=False) for r in outputs}))])
    if rows:
        table(['Caso reservado', 'Esperado', 'Aciertos', 'Respuestas originales'], rows)
    else:
        print('Sin confirmación: no cumplió el criterio en los casos conocidos.')

Ejecución,Temperatura
20261106T203920Z-api-reference,Predeterminada


Etapa,Repetición,Aciertos,Q,V
Conocidos,1,10/12,83.3%,8.3%


Estado,Criterio completo,Llamadas,Tokens,Costo estimado USD
Terminada,No cumple,12,8143,0.001329


Sin confirmación: no cumplió el criterio en los casos conocidos.


Ejecución,Temperatura
20261106T204025Z-api-reference,0.0


Etapa,Repetición,Aciertos,Q,V
Conocidos,1,8/12,66.7%,8.3%


Estado,Criterio completo,Llamadas,Tokens,Costo estimado USD
Terminada,No cumple,12,8143,0.001329


Sin confirmación: no cumplió el criterio en los casos conocidos.


## 6. Resultado del control

- **Qwen 8B con razonamiento:** 12/12 casos conocidos; 24/24 respuestas de confirmación, sin infracciones.
- **Confirmación:** ocho casos reservados, tres repeticiones.
- **Sin cumplir:** barrido directo 8B/32B y controles API.
- **Límite:** política manual fija. Este test no demuestra automejora ni aísla cada ajuste del perfil.

**Continuación:** [lectura conjunta](study-overview.ipynb) con H0, propuestas B1/B2 y tamaño de prompts.
[Historial del Test 4](paired-search.ipynb) · [Ejecución técnica](../utils/execution.md).
